# About Project
- 공학연구인턴십 CWRU

# Cell 1. Best Model 불러오기

In [ ]:
import os
import torch
import pickle
import pandas as pd
import numpy as np
import time
from torch.utils.data import DataLoader
from Model import WDCNN
from utils import (
    set_seed,

    directories, create_directories,

    # 데이터셋 및 학습 관련 함수
    CWRUDataset, train_original_model_with_batch_sizes, 
    fine_tune_unstructured, fine_tune_structured,
    plot_unstructured_pruning_results, plot_structured_pruning_results,
    
    # Pruning 적용 함수
    apply_sparse_training, apply_structured_pruning,
    remove_pruning_and_save_unstructured, remove_pruning_and_save_structured,

    # 평가 함수
    evaluate_classification, measure_inference_time, evaluate_test_performance_structured, evaluate_test_performance_unstructured,
    count_total_parameters, count_nonzero_parameters, evaluate_test_performance_unstructured_avg, 
    evaluate_test_performance_structured_avg,

    # 변수 (경로)
    parameter_dir, original_path, experiment_1_path, experiment_2_path,
    unstructured_path, structured_path,
    
    unstructured_path_100, unstructured_path_1000,
    structured_path_100, structured_path_1000,

    approach_1_path, approach_2_path,
    initialization_approach_1, No_initialization_approach_1,
    initialization_approach_2, No_initialization_approach_2,
    
    unstructured_experiment_1_100_results_path,
    unstructured_experiment_1_1000_results_path,
    structured_experiment_1_100_results_path,
    structured_experiment_1_1000_results_path
)

create_directories(directories)

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ============================================================
# 🔹 Dataset 로드
# ============================================================
train_dataset, val_dataset, test_dataset = (
    CWRUDataset("Train"), CWRUDataset("Validation"), CWRUDataset("Test")
)

# train_loader, val_loader는 어차피 train_original_model_with_batch_sizes 여기서 다시 설정
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, worker_init_fn=lambda _: np.random.seed(42))
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# ============================================================
# 🔹 Original Model 학습 여부 체크 및 로드
# ============================================================
best_model_path = os.path.join(original_path, "best_overall_model.pth")

if not os.path.exists(best_model_path):
    print("\n🔹 No pre-trained model found. Starting training...")
    train_original_model_with_batch_sizes(train_dataset, val_dataset)
else:
    print("\n✅ Pre-trained model found. Skipping training.")

# ✅ Original Model 로드
original_model = WDCNN().to(device)
original_model.load_state_dict(torch.load(best_model_path, map_location=device))

# ✅ Original Model Classification Report & Confusion Matrix
evaluate_classification(original_model, test_loader, device, title="Original Model")


## Cell 1.1 [Optional] 재구현

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report, confusion_matrix
from Model import WDCNN
from utils import CWRUDataset, original_path, set_seed


def evaluate_classification(model, test_loader, device, title="Model Evaluation"):
    """Test Dataset에서 Classification Report 및 Confusion Matrix 출력"""
    model.to(device)
    model.eval()

    y_true, y_pred = [], []
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            predictions = torch.argmax(outputs, dim=1)

            y_true.extend(labels.cpu().numpy())
            y_pred.extend(predictions.cpu().numpy())

    print(f"\n📌 {title} - Classification Report:")
    print(classification_report(y_true, y_pred, target_names=['N', 'IR', 'OR@06', 'B']))

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=['N', 'IR', 'OR@06', 'B'], yticklabels=['N', 'IR', 'OR@06', 'B'])
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.title(f"{title} - Confusion Matrix")
    plt.show()


def train_with_loaded_initial_weights(train_dataset, val_dataset, test_loader, batch_size=256, num_epochs=500, learning_rate=0.01, patience=200):
    """저장된 초기 가중치를 로드하여 재학습 및 성능 재현 검증"""
    set_seed(42)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, worker_init_fn=lambda _: np.random.seed(42))
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    # ✅ 초기 가중치 로드
    init_weights_path = os.path.join(original_path, f"initial_weights_bs{batch_size}.pth")

    if not os.path.exists(init_weights_path):
        raise FileNotFoundError(f"❌ 초기 가중치 파일이 없습니다: {init_weights_path}")

    model = WDCNN().to(device)
    model.load_state_dict(torch.load(init_weights_path, map_location=device))
    print(f"✅ 초기 가중치 로드 완료: {init_weights_path}")

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-3)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=10, verbose=True)

    best_val_acc = 0.0
    best_model_state = None
    early_stop_counter = 0

    for epoch in range(num_epochs):
        model.train()
        train_correct = 0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_correct += (outputs.argmax(dim=1) == labels).sum().item()

        train_acc = train_correct / len(train_loader.dataset)

        # 🔹 Validation
        model.eval()
        val_correct = 0
        val_loss = 0.0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                val_correct += (outputs.argmax(dim=1) == labels).sum().item()

        val_acc = val_correct / len(val_loader.dataset)
        scheduler.step(val_loss)

        print(f"Epoch {epoch+1}: Train Acc: {train_acc:.4f}, Val Acc: {val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_state = model.state_dict()
            early_stop_counter = 0
        else:
            early_stop_counter += 1

        if early_stop_counter >= patience:
            print(f"🛑 Early stopping at epoch {epoch+1}")
            break

    print(f"\n🏆 재학습 완료! 최고 Validation Accuracy: {best_val_acc:.4f}")

    # ✅ 최적 모델 로드 (테스트 평가)
    model.load_state_dict(best_model_state)
    evaluate_classification(model, test_loader, device, title="Reproduced Model")

# ============================================================
# 🔹 Dataset 로드 및 재현 확인 실행
# ============================================================
train_dataset, val_dataset, test_dataset = (
    CWRUDataset("Train"), CWRUDataset("Validation"), CWRUDataset("Test")
)

test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

train_with_loaded_initial_weights(
    train_dataset=train_dataset,
    val_dataset=val_dataset,
    test_loader=test_loader,
    batch_size=256,
    num_epochs=500,
    learning_rate=0.01,
    patience=200
)


# Cell 2. Experiment 1

## Cell 2.1 Unstructured Pruning (100 Epochs)

### Cell 2.1.1 Fine-Tuning

In [ ]:
# ============================================================
# 🔹 Unstructured Pruning Fine-Tuning 수행 (1번만)
# ============================================================
pruning_ratios = [0.2, 0.4, 0.6, 0.8]

unstructured_results_path = unstructured_experiment_1_100_results_path
unstructured_path = unstructured_path_100

# 🔹 저장된 Fine-Tuning 결과가 있는지 확인
unstructured_experiment_1_100_results_path = os.path.join(unstructured_path_100, "unstructured_experiment_results.pkl")
experiment_results = None

if os.path.exists(unstructured_experiment_1_100_results_path):
    with open(unstructured_experiment_1_100_results_path, "rb") as f:
        experiment_results = pickle.load(f)
    print("\n✅ 기존 Fine-Tuning 결과를 불러왔습니다. 학습을 건너뜁니다.")
else:
    print("\n🔹 Starting Unstructured Pruning Fine-Tuning...")
    experiment_results = fine_tune_unstructured(pruning_ratios, train_loader, val_loader, num_epochs=100)
    print("\n✅ Unstructured Pruning Fine-Tuning Completed!")


### Cell 2.1.2 Graph

In [ ]:
# ============================================================
# 🔹 Validation Accuracy 그래프 출력 (Original Model 포함)
# ============================================================
print("\n🔹 Plotting Unstructured Pruning Validation Accuracy Graph...")
plot_unstructured_pruning_results(experiment_results, num_epochs=100, val_loader=val_loader, include_original=True)
print("\n✅ Graph Generated!")


### Cell 2.1.3 Test 성능 평가

In [ ]:
# Test Performance Evaluation
print("\n🔹 Final Test Performance Evaluation for Unstructured Pruning (Including Inference Time)...")
results_df_unstructured = evaluate_test_performance_unstructured(original_model, test_loader, device, experiment_results, unstructured_path)
print("Unstructured Pruning Test Evaluation Results:")
display(results_df_unstructured)

### Cell 2.1.4 Test 성능 평가 (n회 평균)

In [ ]:
# Test Performance Evaluation N회 반복 평균
repeat_times = 100  # 사용자 설정 (원하는 횟수로 변경 가능)

print(f"\n🔹 Final Test Performance Evaluation for Unstructured Pruning (Including Inference Time, Averaged over {repeat_times} runs)...")

results_df_unstructured = evaluate_test_performance_unstructured_avg(
    original_model, test_loader, device, experiment_results, unstructured_path, repeat_times=repeat_times
)

print(f"Unstructured Pruning Test Evaluation Results (Averaged over {repeat_times} runs):")
display(results_df_unstructured)


## Cell 2.2 Unstructured Pruning (1000 Epochs)

### Cell 2.2.1 Fine-Tuning

In [ ]:
# ============================================================
# 🔹 Unstructured Pruning Fine-Tuning 수행 (1번만)
# ============================================================
pruning_ratios = [0.2, 0.4, 0.6, 0.8]

unstructured_results_path = unstructured_experiment_1_1000_results_path
unstructured_path = unstructured_path_1000

# 🔹 저장된 Fine-Tuning 결과가 있는지 확인
experiment_results_path = os.path.join(unstructured_path, "experiment_results.pkl")
experiment_results = None

if os.path.exists(experiment_results_path):
    with open(experiment_results_path, "rb") as f:
        experiment_results = pickle.load(f)
    print("\n✅ 기존 Fine-Tuning 결과를 불러왔습니다. 학습을 건너뜁니다.")
else:
    print("\n🔹 Starting Unstructured Pruning Fine-Tuning...")
    experiment_results = fine_tune_unstructured(pruning_ratios, train_loader, val_loader, num_epochs=1000)
    print("\n✅ Unstructured Pruning Fine-Tuning Completed!")


### Cell 2.2.2 Graph

In [ ]:
# ============================================================
# 🔹 Validation Accuracy 그래프 출력 (Original Model 포함)
# ============================================================
print("\n🔹 Plotting Unstructured Pruning Validation Accuracy Graph...")
plot_unstructured_pruning_results(experiment_results, num_epochs=1000, val_loader=val_loader, include_original=True)
print("\n✅ Graph Generated!")


### Cell 2.2.3 Test 성능 평가

In [ ]:
## ✅ Cell 1.3: Test Performance Evaluation (Inference 포함 - Unstructured)
print("\n🔹 Final Test Performance Evaluation for Unstructured Pruning (Including Inference Time)...")
results_df_unstructured = evaluate_test_performance_unstructured(original_model, test_loader, device, experiment_results, unstructured_path)
print("Unstructured Pruning Test Evaluation Results:")
display(results_df_unstructured)

### Cell 2.2.4 Test 성능 평가 (n회 평균)

In [ ]:
# Test Performance Evaluation N회 반복 평균
repeat_times = 100  # 사용자 설정 (원하는 횟수로 변경 가능)

print(f"\n🔹 Final Test Performance Evaluation for Unstructured Pruning (Including Inference Time, Averaged over {repeat_times} runs)...")

results_df_unstructured = evaluate_test_performance_unstructured_avg(
    original_model, test_loader, device, experiment_results, unstructured_path, repeat_times=repeat_times
)

print(f"Unstructured Pruning Test Evaluation Results (Averaged over {repeat_times} runs):")
display(results_df_unstructured)



## Cell 2.3 Structured Pruning (100 Epochs)

### Cell 2.3.1 Fine-Tuning

In [ ]:
# ============================================================
# 🔹 Structured Pruning Fine-Tuning 수행 (1번만)
# ============================================================
structured_pruning_ratios = [0.2, 0.4, 0.6, 0.8]

structured_results_path = structured_experiment_1_100_results_path
structured_path = structured_path_100

# 🔹 저장된 Fine-Tuning 결과가 있는지 확인
structured_experiment_results_path = os.path.join(structured_path, "structured_experiment_results.pkl")
structured_experiment_results = None

if os.path.exists(structured_experiment_results_path):
    with open(structured_experiment_results_path, "rb") as f:
        structured_experiment_results = pickle.load(f)
    print("\n✅ 기존 Structured Fine-Tuning 결과를 불러왔습니다. 학습을 건너뜁니다.")
else:
    print("\n🔹 Starting Structured Pruning Fine-Tuning...")
    structured_experiment_results = fine_tune_structured(structured_pruning_ratios, train_loader, val_loader, num_epochs=100)
    print("\n✅ Structured Pruning Fine-Tuning Completed!")


### Cell 2.3.2 Graph

In [ ]:
## ✅ Cell 2.2: Validation Accuracy 그래프 출력
print("\n🔹 Plotting Structured Pruning Validation Accuracy Graph...")
plot_structured_pruning_results(structured_experiment_results, num_epochs=100, val_loader=val_loader, include_original=True)
print("\n✅ Graph Generated!")

### Cell 2.3.3 Test 성능 평가

In [ ]:
## ✅ Cell 2.3: Test Performance Evaluation (Inference 포함 - Structured)
print("\n🔹 Final Test Performance Evaluation for Structured Pruning (Including Inference Time)...")
results_df_structured = evaluate_test_performance_structured(original_model, test_loader, device, structured_experiment_results, structured_path)
print("Structured Pruning Test Evaluation Results:")
display(results_df_structured)


### Cell 2.3.4 Test 성능 평가 (n회 평균)

In [ ]:
## ✅ Cell 2.3: Test Performance Evaluation (Inference 포함 - Structured, N회 반복 평균)
repeat_times = 100  # 사용자 설정 (원하는 횟수로 변경 가능)

print(f"\n🔹 Final Test Performance Evaluation for Structured Pruning (Including Inference Time, Averaged over {repeat_times} runs)...")

results_df_structured = evaluate_test_performance_structured_avg(
    original_model, test_loader, device, structured_experiment_results, structured_path, repeat_times=repeat_times
)

print(f"Structured Pruning Test Evaluation Results (Averaged over {repeat_times} runs):")
display(results_df_structured)

## Cell 2.4 Structured Pruning (1000 Epochs)

### Cell 2.4.1 Fine-Tuning

In [ ]:
# ============================================================
# 🔹 Structured Pruning Fine-Tuning 수행 (1번만)
# ============================================================
structured_pruning_ratios = [0.2, 0.4, 0.6, 0.8]

structured_results_path = structured_experiment_1_1000_results_path
structured_path = structured_path_1000

# 🔹 저장된 Fine-Tuning 결과가 있는지 확인
structured_experiment_results_path = os.path.join(structured_path, "structured_experiment_results.pkl")
structured_experiment_results = None

if os.path.exists(structured_experiment_results_path):
    with open(structured_experiment_results_path, "rb") as f:
        structured_experiment_results = pickle.load(f)
    print("\n✅ 기존 Structured Fine-Tuning 결과를 불러왔습니다. 학습을 건너뜁니다.")
else:
    print("\n🔹 Starting Structured Pruning Fine-Tuning...")
    structured_experiment_results = fine_tune_structured(structured_pruning_ratios, train_loader, val_loader, num_epochs=1000)
    print("\n✅ Structured Pruning Fine-Tuning Completed!")


### Cell 2.4.2 Graph

In [ ]:
## ✅ Cell 2.2: Validation Accuracy 그래프 출력
print("\n🔹 Plotting Structured Pruning Validation Accuracy Graph...")
plot_structured_pruning_results(structured_experiment_results, num_epochs=1000, val_loader=val_loader, include_original=True)
print("\n✅ Graph Generated!")

### Cell 2.4.3 Test 성능 평가

In [ ]:
## ✅ Cell 2.3: Test Performance Evaluation (Inference 포함 - Structured)
print("\n🔹 Final Test Performance Evaluation for Structured Pruning (Including Inference Time)...")
results_df_structured = evaluate_test_performance_structured(original_model, test_loader, device, structured_experiment_results, structured_path)
print("Structured Pruning Test Evaluation Results:")
display(results_df_structured)


### Cell 2.4.4 Test 성능 평가 (n회 평균)

In [ ]:
## ✅ Cell 2.3: Test Performance Evaluation (Inference 포함 - Structured, N회 반복 평균)
repeat_times = 100  # 사용자 설정 (원하는 횟수로 변경 가능)

print(f"\n🔹 Final Test Performance Evaluation for Structured Pruning (Including Inference Time, Averaged over {repeat_times} runs)...")

results_df_structured = evaluate_test_performance_structured_avg(
    original_model, test_loader, device, structured_experiment_results, structured_path, repeat_times=repeat_times
)

print(f"Structured Pruning Test Evaluation Results (Averaged over {repeat_times} runs):")
display(results_df_structured)

# Cell 3. Experiment 2

## Cell 3.1 Approach 1

- 가중치 있으면 학습 안하도록 코드 수정!

### Cell 3.1.1 Initialization

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import pickle
import matplotlib.pyplot as plt
from copy import deepcopy
from Model import WDCNN
from utils import remove_pruning_and_save_unstructured, evaluate_classification
import torch.nn.utils.prune as prune

# 경로 설정
original_model_path = r"C:\Users\ChoiSeongHyeon\Desktop\WinningT\Winning_Ticket_Project\CWRU\Parameters\Best Model\best_overall_model.pth"
initial_weights_path = r"C:\Users\ChoiSeongHyeon\Desktop\WinningT\Winning_Ticket_Project\CWRU\Parameters\Best Model\initial_weights_bs256.pth"

# Device 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Percent of Weights Remaining
pruning_steps = [100 * (0.8 ** i) for i in range(30)]

# Dataset 로드
train_dataset, val_dataset, test_dataset = (
    CWRUDataset("Train"), CWRUDataset("Validation"), CWRUDataset("Test")
)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=64, shuffle=False)

val_accs = []
test_accs = []

# 초기 모델 로드 및 초기화 저장
pruned_model = WDCNN().to(device)
pruned_model.load_state_dict(torch.load(best_model_path, map_location=device))
initial_weights = torch.load(best_model_path, map_location=device)

for step, perc in enumerate(pruning_steps):
    print(f"\n🔹 Iteration {step+1}: Percent of Weights Remaining {perc}%")

    # Pruning 적용 (매 Iteration마다 추가 Pruning)
    pruning_amount = 0.2
    for name, module in pruned_model.named_modules():
        if isinstance(module, (nn.Conv1d, nn.Linear)):
            prune.l1_unstructured(module, name="weight", amount=pruning_amount)

    # Mask된 부분에 initial weights 적용
    with torch.no_grad():
        for name, module in pruned_model.named_modules():
            if isinstance(module, (nn.Conv1d, nn.Linear)) and hasattr(module, 'weight_mask'):
                init_param = initial_weights[f"{name}.weight"]
                mask = module.weight_mask
                module.weight_orig.copy_(init_param * mask)

    # Fine-tuning
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(pruned_model.parameters(), lr=1e-5)

    best_val_acc = 0.0
    best_pruned_model = None

    for epoch in range(100):
        pruned_model.train()
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = pruned_model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

        # Validation
        pruned_model.eval()
        val_correct = 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = pruned_model(inputs)
                val_correct += (outputs.argmax(dim=1) == labels).sum().item()
        val_acc = val_correct / len(val_dataset)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_pruned_model = deepcopy(pruned_model)

    # 최적 모델 저장
    model_save_path = os.path.join(initialization_approach_1, f"pruned_{perc:.６f}.pth")
    remove_pruning_and_save_unstructured(best_pruned_model, model_save_path)

    # Validation Accuracy 기록
    val_accs.append(best_val_acc)

    # Test Accuracy 평가
    best_pruned_model.eval()
    test_correct = 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = best_pruned_model(inputs)
            test_correct += (outputs.argmax(dim=1) == labels).sum().item()
    test_acc = test_correct / len(test_dataset)
    test_accs.append(test_acc)

# 결과 시각화
plt.figure(figsize=(8, 6))
plt.plot(pruning_steps, test_accs, marker='o', linestyle='-')
plt.xlabel('Percent of Weights Remaining')
plt.ylabel('Test Accuracy')
plt.title('Test Accuracy over Pruning Iterations')
plt.gca().invert_xaxis()
plt.grid(True)
plt.show()

# Test Accuracy 표 출력
import pandas as pd
df = pd.DataFrame({
    'Percent of Weights Remaining': pruning_steps,
    'Validation Accuracy': val_accs,
    'Test Accuracy': test_accs
})
print(df)
display(df)


### Cell 3.1.2 No Initialization

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from copy import deepcopy
from Model import WDCNN
from utils import remove_pruning_and_save_unstructured, CWRUDataset
import torch.nn.utils.prune as prune

# Dataset 로드
train_dataset, val_dataset, test_dataset = (
    CWRUDataset("Train"), CWRUDataset("Validation"), CWRUDataset("Test")
)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=64, shuffle=False)

val_accs = []
test_accs = []

# 초기 모델 로드 및 초기화 저장
pruned_model = WDCNN().to(device)
pruned_model.load_state_dict(torch.load(original_model_path, map_location=device))

for step, perc in enumerate(pruning_steps):
    print(f"\n🔹 Iteration {step+1}: Percent of Weights Remaining {perc}%")

    # Pruning 적용 (매 Iteration마다 추가 Pruning)
    pruning_amount = 0.2
    for name, module in pruned_model.named_modules():
        if isinstance(module, (nn.Conv1d, nn.Linear)):
            prune.l1_unstructured(module, name="weight", amount=pruning_amount)

    # Fine-tuning
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(pruned_model.parameters(), lr=1e-5)

    best_val_acc = 0.0
    best_pruned_model = None

    for epoch in range(100):
        pruned_model.train()
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = pruned_model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

        # Validation
        pruned_model.eval()
        val_correct = 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = pruned_model(inputs)
                val_correct += (outputs.argmax(dim=1) == labels).sum().item()
        val_acc = val_correct / len(val_dataset)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_pruned_model = deepcopy(pruned_model)

    # 최적 모델 저장
    model_save_path = os.path.join(No_initialization_approach_1, f"pruned_{perc:.6f}.pth")
    remove_pruning_and_save_unstructured(best_pruned_model, model_save_path)

    # Validation Accuracy 기록
    val_accs.append(best_val_acc)

    # Test Accuracy 평가
    best_pruned_model.eval()
    test_correct = 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = best_pruned_model(inputs)
            test_correct += (outputs.argmax(dim=1) == labels).sum().item()
    test_acc = test_correct / len(test_dataset)
    test_accs.append(test_acc)

# 결과 시각화
plt.figure(figsize=(8, 6))
plt.plot(pruning_steps, test_accs, marker='o', linestyle='-')
plt.xlabel('Percent of Weights Remaining')
plt.ylabel('Test Accuracy')
plt.title('CWRU Dataset - Test Accuracy over Pruning Iterations (No Initialization)')
plt.gca().invert_xaxis()
plt.grid(True)
plt.show()

# Test Accuracy 표 출력
import pandas as pd
df = pd.DataFrame({
    'Percent of Weights Remaining': pruning_steps,
    'Validation Accuracy': val_accs,
    'Test Accuracy': test_accs
})
print(df)
display(df)

## Cell 3.2 Approach 2
 - 가중치 존재해도 다시 학습하게 코드 되어 있음

### Cell 3.2.1 Initialization

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import pickle
import matplotlib.pyplot as plt
from copy import deepcopy
from Model import WDCNN
from utils import remove_pruning_and_save_unstructured, evaluate_classification
import torch.nn.utils.prune as prune

# Dataset 로드
train_dataset, val_dataset, test_dataset = (
    CWRUDataset("Train"), CWRUDataset("Validation"), CWRUDataset("Test")
)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=64, shuffle=False)

val_accs = []
test_accs = []

# 초기 모델 로드 및 초기화 저장
pruned_model = WDCNN().to(device)
initial_weights = deepcopy(pruned_model.state_dict())

# 초기 가중치를 iterative_save_path에 저장
initial_weights_save_path = os.path.join(initialization_approach_2, 'initial_weights.pth')
torch.save(initial_weights, initial_weights_save_path)

for step, perc in enumerate(pruning_steps):
    print(f"\n🔹 Iteration {step+1}: Percent of Weights Remaining {perc}%")

    # Pruning 적용 (매 Iteration마다 추가 Pruning)
    pruning_amount = 0.2
    for name, module in pruned_model.named_modules():
        if isinstance(module, (nn.Conv1d, nn.Linear)):
            prune.l1_unstructured(module, name="weight", amount=pruning_amount)

    # Mask된 부분에 initial weights 적용
    initial_weights = torch.load(initial_weights_save_path, map_location=device)
    with torch.no_grad():
        for name, module in pruned_model.named_modules():
            if isinstance(module, (nn.Conv1d, nn.Linear)) and hasattr(module, 'weight_mask'):
                init_param = initial_weights[f"{name}.weight"]
                mask = module.weight_mask
                module.weight_orig.copy_(init_param * mask)

    # Fine-tuning
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(pruned_model.parameters(), lr=1e-5)

    best_val_acc = 0.0
    best_pruned_model = None

    for epoch in range(100):
        pruned_model.train()
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = pruned_model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

        # Validation
        pruned_model.eval()
        val_correct = 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = pruned_model(inputs)
                val_correct += (outputs.argmax(dim=1) == labels).sum().item()
        val_acc = val_correct / len(val_dataset)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_pruned_model = deepcopy(pruned_model)

    # 최적 모델 저장
    model_save_path = os.path.join(initialization_approach_2, f"pruned_{perc:.6f}.pth")
    remove_pruning_and_save_unstructured(best_pruned_model, model_save_path)

    # Validation Accuracy 기록
    val_accs.append(best_val_acc)

    # Test Accuracy 평가
    best_pruned_model.eval()
    test_correct = 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = best_pruned_model(inputs)
            test_correct += (outputs.argmax(dim=1) == labels).sum().item()
    test_acc = test_correct / len(test_dataset)
    test_accs.append(test_acc)

# 결과 시각화
plt.figure(figsize=(8, 6))
plt.plot(pruning_steps, test_accs, marker='o', linestyle='-')
plt.xlabel('Percent of Weights Remaining')
plt.ylabel('Test Accuracy')
plt.title('Test Accuracy over Pruning Iterations')
plt.gca().invert_xaxis()
plt.grid(True)
plt.show()

# Test Accuracy 표 출력
import pandas as pd
df = pd.DataFrame({
    'Percent of Weights Remaining': pruning_steps,
    'Validation Accuracy': val_accs,
    'Test Accuracy': test_accs
})
print(df)
display(df)


### Cell 3.2.2 No Initialization

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import pickle
import matplotlib.pyplot as plt
from copy import deepcopy
from Model import WDCNN
from utils import remove_pruning_and_save_unstructured, evaluate_classification
import torch.nn.utils.prune as prune


# Dataset 로드
train_dataset, val_dataset, test_dataset = (
    CWRUDataset("Train"), CWRUDataset("Validation"), CWRUDataset("Test")
)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=64, shuffle=False)

val_accs = []
test_accs = []

# 초기 모델 로드 및 초기화 저장
pruned_model = WDCNN().to(device)
initial_weights = deepcopy(pruned_model.state_dict())

for step, perc in enumerate(pruning_steps):
    print(f"\n🔹 Iteration {step+1}: Percent of Weights Remaining {perc}%")

    if step == 0:
        # 초기 모델 학습
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(pruned_model.parameters(), lr=1e-5)

        best_val_acc = 0.0
        best_pruned_model = None

        for epoch in range(100):
            pruned_model.train()
            for inputs, labels in train_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                optimizer.zero_grad()
                outputs = pruned_model(inputs)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()

            # Validation
            pruned_model.eval()
            val_correct = 0
            with torch.no_grad():
                for inputs, labels in val_loader:
                    inputs, labels = inputs.to(device), labels.to(device)
                    outputs = pruned_model(inputs)
                    val_correct += (outputs.argmax(dim=1) == labels).sum().item()
            val_acc = val_correct / len(val_dataset)

            if val_acc > best_val_acc:
                best_val_acc = val_acc
                best_pruned_model = deepcopy(pruned_model)

        pruned_model = best_pruned_model
    else:
        # Pruning 적용 (매 Iteration마다 추가 Pruning)
        pruning_amount = 0.2
        for name, module in pruned_model.named_modules():
            if isinstance(module, (nn.Conv1d, nn.Linear)):
                prune.l1_unstructured(module, name="weight", amount=pruning_amount)

    # Fine-tuning (Step 0에서는 이미 학습 완료 상태이므로 이후 Iteration만 Fine-tuning)
    if step > 0:
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(pruned_model.parameters(), lr=1e-5)

        best_val_acc = 0.0
        best_pruned_model = None

        for epoch in range(100):
            pruned_model.train()
            for inputs, labels in train_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                optimizer.zero_grad()
                outputs = pruned_model(inputs)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()

            # Validation
            pruned_model.eval()
            val_correct = 0
            with torch.no_grad():
                for inputs, labels in val_loader:
                    inputs, labels = inputs.to(device), labels.to(device)
                    outputs = pruned_model(inputs)
                    val_correct += (outputs.argmax(dim=1) == labels).sum().item()
            val_acc = val_correct / len(val_dataset)

            if val_acc > best_val_acc:
                best_val_acc = val_acc
                best_pruned_model = deepcopy(pruned_model)

        pruned_model = best_pruned_model

    # 최적 모델 저장
    model_save_path = os.path.join(No_initialization_approach_2, f"pruned_{perc:.6f}.pth")
    if step > 0:
        remove_pruning_and_save_unstructured(pruned_model, model_save_path)
    else:
        torch.save(pruned_model.state_dict(), model_save_path)


    # Validation Accuracy 기록
    val_accs.append(best_val_acc)

    # Test Accuracy 평가
    pruned_model.eval()
    test_correct = 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = pruned_model(inputs)
            test_correct += (outputs.argmax(dim=1) == labels).sum().item()
    test_acc = test_correct / len(test_dataset)
    test_accs.append(test_acc)

# 결과 시각화
plt.figure(figsize=(8, 6))
plt.plot(pruning_steps, test_accs, marker='o', linestyle='-')
plt.xlabel('Percent of Weights Remaining')
plt.ylabel('Test Accuracy')
plt.title('Test Accuracy over Pruning Iterations')
plt.gca().invert_xaxis()
plt.grid(True)
plt.show()

# Test Accuracy 표 출력
import pandas as pd
df = pd.DataFrame({
    'Percent of Weights Remaining': pruning_steps,
    'Validation Accuracy': val_accs,
    'Test Accuracy': test_accs
})
print(df)
display(df)
